In [1]:
!pip -q install transformers datasets evaluate peft sentencepiece accelerate safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_from_disk, Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ===============================
# DATA
# ===============================
TRAIN_DS_PATH = "/content/drive/MyDrive/byt5_cache/train_dataset"
VAL_DS_PATH   = "/content/drive/MyDrive/byt5_cache/val_dataset"

# ===============================
# XLM-R LARGE MODEL
# ===============================
XLMR_OUTPUT_DIR = "/content/drive/MyDrive/xlmr_large_dual_head_model_try_2"
XLMR_MODEL_DIR = os.path.join(XLMR_OUTPUT_DIR, "best_model")
ROOT_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2root.json")
SUFFIX_MAP_PATH = os.path.join(XLMR_OUTPUT_DIR, "id2suffix.json")

# ===============================
# OUTPUT DATASETS WITH HINTS
# ===============================
HINT_TRAIN_SAVE_PATH = "/content/drive/MyDrive/mt5_hint_cache/train_dataset_with_hints"
HINT_VAL_SAVE_PATH   = "/content/drive/MyDrive/mt5_hint_cache/val_dataset_with_hints"

# ===============================
# OUTPUT MODEL
# ===============================
MT5_HINT_OUTPUT_DIR = "/content/drive/MyDrive/mt5_hint_training_output"
MT5_HINT_MERGED_DIR = "/content/drive/MyDrive/mt5_hint_merged_model"

# ===============================
# MODEL NAMES
# ===============================
MT5_MODEL_NAME = "google/mt5-base"
XLMR_MODEL_NAME = "xlm-roberta-large"

# ===============================
# TRAINING SETTINGS
# ===============================
MAX_INPUT_LEN = 256
MAX_TARGET_LEN = 256

BATCH_SIZE = 8
NUM_EPOCHS = 4
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [5]:
train_dataset = load_from_disk(TRAIN_DS_PATH)
val_dataset = load_from_disk(VAL_DS_PATH)

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

print(train_dataset[0].keys())
print(train_dataset[0])

Train size: 20717
Val size: 5180
dict_keys(['expected_script', 'generated_script'])
{'expected_script': 'எனக்கு இந்த software-ஐ use பண்ண shortcut keys சொல்லுங்க.', 'generated_script': 'எனக்கு இந்த sofa-அ  use பண்ண short cut key சொல்லுங்க'}


In [6]:
def clean_token(tok):
    tok = str(tok).strip()
    tok = tok.strip(".,!?;:\"“”‘’()[]{}")
    return tok

def tokenize(text):
    return [clean_token(tok) for tok in str(text).strip().split() if clean_token(tok)]

def is_english_word(token):
    return bool(re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", token))

def is_tamil_text(text):
    return bool(re.search(r"[\u0B80-\u0BFF]", text))

def is_mixed_token(token):
    if "-" not in token:
        return False
    parts = token.split("-", 1)
    if len(parts) != 2:
        return False
    left, right = parts[0].strip(), parts[1].strip()
    return is_english_word(left) and is_tamil_text(right)

def get_token_class(token):
    if is_mixed_token(token):
        return "MIX"
    elif is_english_word(token):
        return "EN"
    elif is_tamil_text(token):
        return "TA"
    else:
        return "OTHER"

def split_mixed_token(token):
    if "-" not in token:
        return None, None
    left, right = token.split("-", 1)
    return left.strip(), right.strip()

def extract_root_suffix(token, token_class):
    if token_class == "MIX":
        root, suffix = split_mixed_token(token)
        return root if root else "", suffix if suffix else ""
    elif token_class == "EN":
        return token, "NULL"
    return "", ""

def build_model_input(left_context, token, right_context, token_class, root, suffix):
    return f"LEFT={left_context} TOKEN={token} RIGHT={right_context} CLASS={token_class} ROOT={root} SUFFIX={suffix}"

def get_context(tokens, idx, window=2):
    left_tokens = tokens[max(0, idx-window):idx]
    right_tokens = tokens[idx+1:idx+1+window]
    return " ".join(left_tokens).strip(), " ".join(right_tokens).strip()

In [7]:
with open(ROOT_MAP_PATH, "r", encoding="utf-8") as f:
    id2root = json.load(f)

with open(SUFFIX_MAP_PATH, "r", encoding="utf-8") as f:
    id2suffix = json.load(f)

id2root = {int(k): v for k, v in id2root.items()}
id2suffix = {int(k): v for k, v in id2suffix.items()}

print("Root labels:", len(id2root))
print("Suffix labels:", len(id2suffix))

Root labels: 794
Suffix labels: 62


In [8]:
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_MODEL_NAME)

class XLMRDualHeadModel(nn.Module):
    def __init__(self, model_name, num_root_labels, num_suffix_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.root_classifier = nn.Linear(hidden_size, num_root_labels)
        self.suffix_classifier = nn.Linear(hidden_size, num_suffix_labels)

    def forward(self, input_ids=None, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]
        cls_repr = self.dropout(cls_repr)

        root_logits = self.root_classifier(cls_repr)
        suffix_logits = self.suffix_classifier(cls_repr)

        return {
            "root_logits": root_logits,
            "suffix_logits": suffix_logits
        }

xlmr_model = XLMRDualHeadModel(
    model_name=XLMR_MODEL_NAME,
    num_root_labels=len(id2root),
    num_suffix_labels=len(id2suffix)
)

state_dict_path = os.path.join(XLMR_MODEL_DIR, "model.safetensors")

if os.path.exists(state_dict_path):
    from safetensors.torch import load_file
    state_dict = load_file(state_dict_path)
    xlmr_model.load_state_dict(state_dict)
else:
    state_dict_path = os.path.join(XLMR_MODEL_DIR, "pytorch_model.bin")
    state_dict = torch.load(state_dict_path, map_location="cpu")
    xlmr_model.load_state_dict(state_dict)

xlmr_model.to(device)
xlmr_model.eval()

print("Loaded trained XLM-R large model.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded trained XLM-R large model.


In [9]:
MAX_LEN_XLMR = 96

def predict_token_with_confidence(tokens, idx):
    token = tokens[idx]
    token_class = get_token_class(token)

    if token_class not in ["EN", "MIX"]:
        return {
            "original_token": token,
            "token_class": token_class,
            "pred_root": None,
            "pred_suffix": None,
            "root_conf": None,
            "suffix_conf": None,
            "corrected_token": token
        }

    wrong_root, wrong_suffix = extract_root_suffix(token, token_class)
    left_context, right_context = get_context(tokens, idx, window=2)

    model_input = build_model_input(
        left_context=left_context,
        token=token,
        right_context=right_context,
        token_class=token_class,
        root=wrong_root,
        suffix=wrong_suffix
    )

    enc = xlmr_tokenizer(
        model_input,
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN_XLMR,
        return_tensors="pt"
    )

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        outputs = xlmr_model(input_ids=input_ids, attention_mask=attention_mask)
        root_probs = F.softmax(outputs["root_logits"], dim=1)
        suffix_probs = F.softmax(outputs["suffix_logits"], dim=1)

        root_pred_id = torch.argmax(root_probs, dim=1).item()
        suffix_pred_id = torch.argmax(suffix_probs, dim=1).item()

        root_conf = root_probs[0, root_pred_id].item()
        suffix_conf = suffix_probs[0, suffix_pred_id].item()

    pred_root = id2root[root_pred_id]
    pred_suffix = id2suffix[suffix_pred_id]

    if token_class == "EN":
        corrected_token = pred_root
    else:
        corrected_token = f"{pred_root}-{pred_suffix}"

    return {
        "original_token": token,
        "token_class": token_class,
        "wrong_root": wrong_root,
        "wrong_suffix": wrong_suffix,
        "pred_root": pred_root,
        "pred_suffix": pred_suffix,
        "root_conf": root_conf,
        "suffix_conf": suffix_conf,
        "corrected_token": corrected_token
    }

In [10]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio()

def is_case_only_change(src, tgt):
    return str(src).lower() == str(tgt).lower() and str(src) != str(tgt)

ROOT_CONF_THRESH_MIX_HINT = 0.75
SUFFIX_CONF_THRESH_MIX_HINT = 0.65
ROOT_SIMILARITY_THRESH = 0.65

def should_add_hint(pred_info):
    token_class = pred_info["token_class"]
    if token_class != "MIX":
        return False

    original_token = pred_info["original_token"]
    corrected_token = pred_info["corrected_token"]

    if corrected_token == original_token:
        return False

    wrong_root = pred_info["wrong_root"]
    wrong_suffix = pred_info["wrong_suffix"]
    pred_root = pred_info["pred_root"]
    pred_suffix = pred_info["pred_suffix"]
    root_conf = pred_info["root_conf"]
    suffix_conf = pred_info["suffix_conf"]

    if root_conf is None or suffix_conf is None:
        return False

    if is_case_only_change(original_token, corrected_token):
        return False
    if is_case_only_change(wrong_root, pred_root) and wrong_suffix == pred_suffix:
        return False

    # suffix-only correction
    if wrong_root.lower() == pred_root.lower() and wrong_suffix != pred_suffix:
        return suffix_conf >= 0.80

    # root-changing correction
    sim = similarity(wrong_root, pred_root)

    if wrong_root.lower() != pred_root.lower():
        if root_conf < ROOT_CONF_THRESH_MIX_HINT:
            return False
        if suffix_conf < SUFFIX_CONF_THRESH_MIX_HINT:
            return False
        if sim < ROOT_SIMILARITY_THRESH:
            return False
        if abs(len(wrong_root) - len(pred_root)) > 3:
            return False
        return True

    return False

In [11]:
def build_hints_for_sentence(asr_sentence):
    tokens = tokenize(asr_sentence)
    hints = []

    for idx in range(len(tokens)):
        pred_info = predict_token_with_confidence(tokens, idx)
        apply_hint = should_add_hint(pred_info)

        if apply_hint:
            original_token = pred_info["original_token"]
            corrected_token = pred_info["corrected_token"]
            hints.append(f"{original_token} -> {corrected_token}")

    hint_text = " ; ".join(hints)
    return hint_text

In [12]:
def add_hint_field(example):
    generated_text = str(example["generated_script"]).strip()
    hint_text = build_hints_for_sentence(generated_text)
    example["hint_text"] = hint_text
    return example

train_with_hints = train_dataset.map(add_hint_field)
val_with_hints = val_dataset.map(add_hint_field)

print(train_with_hints[0])

Map:   0%|          | 0/20717 [00:00<?, ? examples/s]

Map:   0%|          | 0/5180 [00:00<?, ? examples/s]

{'expected_script': 'எனக்கு இந்த software-ஐ use பண்ண shortcut keys சொல்லுங்க.', 'generated_script': 'எனக்கு இந்த sofa-அ  use பண்ண short cut key சொல்லுங்க', 'hint_text': ''}


In [13]:
os.makedirs(os.path.dirname(HINT_TRAIN_SAVE_PATH), exist_ok=True)

train_with_hints.save_to_disk(HINT_TRAIN_SAVE_PATH)
val_with_hints.save_to_disk(HINT_VAL_SAVE_PATH)

print("Saved train_with_hints to:", HINT_TRAIN_SAVE_PATH)
print("Saved val_with_hints to:", HINT_VAL_SAVE_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/20717 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5180 [00:00<?, ? examples/s]

Saved train_with_hints to: /content/drive/MyDrive/mt5_hint_cache/train_dataset_with_hints
Saved val_with_hints to: /content/drive/MyDrive/mt5_hint_cache/val_dataset_with_hints


In [14]:
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_NAME)

def preprocess_with_hints(example):
    generated_text = str(example["generated_script"]).strip()
    hint_text = str(example["hint_text"]).strip()
    target_text = str(example["expected_script"]).strip()

    if hint_text:
        input_text = f"fix tamil-english: {generated_text} [HINTS: {hint_text}]"
    else:
        input_text = f"fix tamil-english: {generated_text}"

    model_inputs = mt5_tokenizer(
        input_text,
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = mt5_tokenizer(
        target_text,
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_with_hints.map(preprocess_with_hints)
val_tok = val_with_hints.map(preprocess_with_hints)

print(train_tok[0].keys())

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

Map:   0%|          | 0/20717 [00:00<?, ? examples/s]

Map:   0%|          | 0/5180 [00:00<?, ? examples/s]

dict_keys(['expected_script', 'generated_script', 'hint_text', 'input_ids', 'attention_mask', 'labels'])


In [15]:
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT5_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v", "o"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

mt5_model = get_peft_model(mt5_model, lora_config)
mt5_model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 970,112,256 || trainable%: 0.3648


In [17]:
!pip -q install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.2 MB/s eta 0:00:00


In [23]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=mt5_tokenizer,
    model=mt5_model
)

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # convert to numpy
    preds = np.array(preds)
    labels = np.array(labels)

    # replace invalid prediction ids
    preds = np.where(preds < 0, mt5_tokenizer.pad_token_id, preds)

    # replace -100 in labels
    labels = np.where(labels != -100, labels, mt5_tokenizer.pad_token_id)

    # cast safely
    preds = preds.astype(np.int64)
    labels = labels.astype(np.int64)

    decoded_preds = mt5_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = mt5_tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    wer_score = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    cer_score = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "wer": wer_score,
        "cer": cer_score
    }

In [24]:
training_args = Seq2SeqTrainingArguments(
    output_dir=MT5_HINT_OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=200,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    fp16=False,
    bf16=True if device == "cuda" else False,

    report_to="none"
)

In [25]:
trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [26]:
train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.380761,1.198963,0.279313,0.079362
2,1.343381,1.123530,0.279265,0.082355
3,1.297859,1.089708,0.275621,0.082122
4,1.233762,1.082668,0.274120,0.081315


TrainOutput(global_step=10360, training_loss=1.3561295719220372, metrics={'train_runtime': 5362.4922, 'train_samples_per_second': 15.453, 'train_steps_per_second': 1.932, 'total_flos': 6142360228485120.0, 'train_loss': 1.3561295719220372, 'epoch': 4.0})


In [27]:
trainer.save_model(MT5_HINT_OUTPUT_DIR)
mt5_tokenizer.save_pretrained(MT5_HINT_OUTPUT_DIR)

print("Saved hint-trained LoRA model to:", MT5_HINT_OUTPUT_DIR)

Saved hint-trained LoRA model to: /content/drive/MyDrive/mt5_hint_training_output


In [28]:
merged_model = mt5_model.merge_and_unload()
merged_model.save_pretrained(MT5_HINT_MERGED_DIR)
mt5_tokenizer.save_pretrained(MT5_HINT_MERGED_DIR)

print("Saved merged hint-trained mT5 model to:", MT5_HINT_MERGED_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved merged hint-trained mT5 model to: /content/drive/MyDrive/mt5_hint_merged_model


In [ ]:
eval_results = trainer.evaluate()
print(eval_results)